## Deliverable 2

### Jon Gundesø og Jonas Kristoffersen

### Task 1

In [31]:
import re
from pathlib import Path


def file_reader(path_name: str) -> tuple[str, str]:
    """It reads a file and return a tuple with encoding and text"""
    # we have to read as bytes so that python doenst decode the file
    file = path_name.read_bytes()
    # we check the encoding
    try:
        encoding = "utf-8"
        text = file.decode(encoding)
    except UnicodeDecodeError:
        encoding = "iso-8859-1"
        

    # we read the context of the file using the encodeing we found
    text = path_name.read_text(encoding=encoding)
    return text, encoding

def log_parser(text : str) -> list[dict["timestamp" : str, "register_id" : str, "item" : str, "price" : float, "currency" : str, "status" : str]]:
    parsed_logs = []
    """Parsers logs into into the 6 key fields and remove cancelled transection"""
    # first we need to splitt the transaction record into individual transactions
    not_wanted = "--- PRINTER PAPER LOW ---\n|=== NETWORK SYNC OK ===\n"
    clean_text = re.sub(not_wanted, "", text)
    transactions = re.split("\n", clean_text)
    # then we get the data we want for each line
    for i in transactions:
        # got some help from ai to make a pattern that groups all the values we want
        monster = r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] \[(?P<register_id>POS_\d+)\] item=(?P<item>\w+) price=(?P<price>[\d\.]+) (?P<currency>\w+) \((?P<status>\w+)\)"
        # we search for our pattern in the string
        match = re.search(monster, i)
        # we check if it matches
        if match:
            # we add the group dictionary to the list
            parsed_logs.append(match.groupdict())

    filtered_list = [d for d in parsed_logs if d.get("status") != "CANCELLED"]

    return filtered_list


### Task 2

In [32]:
import json


def find_log_files(path: str | Path) -> list[Path]:
    """Find all .log files in a directory and its subdirectories."""
    path = Path(path)
    log_files = []

    for item in path.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


def find_activeID(data: dict | list) -> set[str]:
    """Find all register IDs where active is True."""

    active_registers = set()

    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_activeID(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_activeID(item))

    return active_registers


### Task 3

In [33]:
import numpy as np
import pandas as pd


def remove_outliers(df: pd.DataFrame) -> np.array:
    """The function remove value errors and values who is bigger than 3x SD"""
    # make a list for valid values to make a median value and a list that is a copy of the log
    x = []
    y = []

    for i in df:
        values = float(i["price"])
        if values < 900:
            # appends the values to both lists if the value is over 900
            x.append(values)
            y.append(values)
            
        else:
            # appends the value to the list with all values
            y.append(values)

    # calculate mean and standard deviation
    mean = np.mean(x)
    sigma = np.std(x)

    # make the list with all values into an array
    arr = np.array(y)
    # swap the error values for the mean value
    arr[arr == 999] = mean
    # swap the values that are more than 3 standard deviations away
    arr[arr - mean > 3 * sigma] = mean
    return arr



def compute_refund_ratio(succes_count, refund_count) -> np.ndarray:
    """"Compute refund ration. Caount sales and refund inside the code, 
    because we miss a function in the deliverable task."""
    # known values
    e = 10e-6

    # calculate the refund ratio
    Refund_ratio = (refund_count / succes_count + e)
    return Refund_ratio





### Task 4

In [34]:
import pandas as pd


def transactions_to_dataframe(transactions):
    """Convert transactions to a chronologically sorted DataFrame."""
    df = pd.DataFrame(transactions)
    if df.empty:
        return df
    
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    return df


def hourly_revenue_by_item(df):
    """Calculate total hourly revenue for each item."""
    
    hourly_revenue = (
        df.groupby("item")["price"]
        .resample("1h")
        .sum()
    )

    return hourly_revenue


def make_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Create a pivot table showing total revenue by item and status."""

    pivot = pd.pivot_table(
        df,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

    return pivot


def refund_ratio_by_item(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item."""

    items = sorted(df["item"].unique())

    success_count = (
        df[df["status"] == "SUCCESS"]["item"]
        .value_counts()
        .reindex(items, fill_value=0)
        .to_numpy(dtype=float)
    )

    refund_count = (
        df[df["status"] == "REFUND"]["item"]
        .value_counts()
        .reindex(items, fill_value=0)
        .to_numpy(dtype=float)
    )

    refund_ratios = compute_refund_ratio(success_count, refund_count)

    return pd.DataFrame({
        "item": items,
        "refund_ratio": refund_ratios
    })

### Task 5

In [38]:
# 1. Find log files
log_files = find_log_files("d2_data")
    
print("=== Log files found ===")
for log_file in log_files:
    print(log_file)

# 1.2. Find active id
with open("d2_data\store_config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

active_id = find_activeID(config)

print("\n=== Active registers ===")
for register_id in active_id:
    print(register_id)

# 2. Read log files and parse transactions
transactions = []

for log_file in log_files:
    text, encoding = file_reader(log_file)
    parsed_logs = log_parser(text)
    transactions.extend(parsed_logs)

print(f"\nNumber of transactions found: {len(transactions)}")

print(transactions)

# 3.
df = transactions_to_dataframe(transactions)

print(df)
# 4.
df["price"] = remove_outliers(df["price"])

# 5.
df.loc[df["currency"] == "NOK", "price"] *= 10.0
            

    
    
# 6. Create summary files
hourly = hourly_revenue_by_item(df)
pivot = make_pivot(df)
refund_ratios = refund_ratio_by_item(df)

hourly.to_json("hourly_revenue.json")
pivot.to_csv("status_pivot_table.csv")
refund_ratios.to_csv("item_refund_ratios.csv")


print("\nTransactions")
print(df.head())

print("\nHourly revenue")
print(hourly.head())

print("\nStatus pivot table")
print(pivot)

print("\nRefund ratios")
print(refund_ratios)


=== Log files found ===
d2_data\2026\library_cafe\POS_01\register_sales.log
d2_data\2026\library_cafe\POS_02\register_sales.log
d2_data\2026\library_cafe\POS_03\register_sales.log
d2_data\2026\science_building\POS_01\register_sales.log
d2_data\2026\science_building\POS_02\register_sales.log
d2_data\2026\science_building\POS_03\register_sales.log
d2_data\2026\student_union\POS_01\register_sales.log
d2_data\2026\student_union\POS_02\register_sales.log
d2_data\2026\student_union\POS_03\register_sales.log

=== Active registers ===
POS_03
POS_02
POS_01

Number of transactions found: 587
[{'timestamp': '2026-10-15 07:02:52', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': '4.69', 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:06:18', 'register_id': 'POS_01', 'item': 'Latte', 'price': '4.69', 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:09:08', 'register_id': 'POS_01', 'item': 'Muffin', 'price': '3.37', 'currency': 'NOK', 'status': 'SUCC

<>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\jonas\AppData\Local\Temp\ipykernel_18360\820875322.py:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  with open("d2_data\store_config.json", "r", encoding="utf-8") as file:


TypeError: string indices must be integers, not 'str'